# Level 3: replay a recorded context and read the `answer` probability

Runtime: Colab, GPU **T4**, fp16. Nothing is published or trained here; the notebook only loads two checkpoints and reads their next-token distribution on frozen contexts.

**Read the result honestly.** The merged model lowers `answer` on a false-success context *and* on true-finish contexts: it is a causal lever on stopping, not a selective repair. No score gain is claimed (+1.9 pt, CI95 [-4.3 ; +8.0], 165 tasks).

In [ ]:
!pip -q install 'transformers>=5' accelerate pillow huggingface_hub
!git clone https://github.com/martinbon39/cogito-ergo-feci 2>/dev/null || true
%cd cogito-ergo-feci/reproduce

In [ ]:
!python probe_decision.py --dry-run   # checks the message construction, loads no model

In [ ]:
# Original Holo vs the merged checkpoint. Replace HF_REPO_ID once the merged weights are on the Hub;
# until then, build it yourself (level 2): python rebuild_merge.py --build /content/merged  and pass /content/merged.
MERGED = 'HF_REPO_ID'
HOLO = 'Hcompany/Holo-3.1-4B@8c88265a5a159bfd1492db9243733dd2e6e04a6e'
!python probe_decision.py --models {HOLO} {MERGED}

In [ ]:
# Optional and slow on a T4: sample N generations per context (the measure behind 8/8 -> 3/8).
!python probe_decision.py --models {HOLO} {MERGED} --sample 4 --contexts ctx1_false_success_feasible ctx3_true_finish_multi_apps

## What to expect

- `ctx1` (false success, feasible): our recorded sampling, 8 samples, T=0.8: original 8/8 `answer`, merge beta=0.5 3/8.
- `ctx2` (infeasible task, `answer` is the correct action): 8/8 and 8/8.
- `ctx3`, `ctx4` (true finishes): original 8/8, merged 1/8 on each. The effect is **not selective**.
- The teacher-forced P(answer) is read after the model has already written "task completed": it is mostly confidence in copying a written conclusion, and is expected to stay high for both models. Run the sampled cell for the informative number: the default output alone will show little or no difference.
